In [17]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [18]:
movie_data = pd.read_csv("../../datasets/processed/movie_data.csv")

movie_data["content"] = movie_data["content"].fillna("")
movie_data = movie_data.drop_duplicates(subset="movieId")
movie_data = movie_data.reset_index(drop=True)

print("movie_data shape after dedup:", movie_data.shape)
movie_data.head()

movie_data shape after dedup: (3650, 11)


,userId,movieId,rating,timestamp,title,genres,year,clean_title,average_rating,rating_count,content
0,1,1,4.0,964982703,Toy Story (1995),adventure animation children comedy fantasy,1995,Toy Story,3.920930,215,Toy Story adventure animation children comedy ...
1,1,3,4.0,964981247,Grumpier Old Men (1995),comedy romance,1995,Grumpier Old Men,3.259615,52,Grumpier Old Men comedy romance
2,1,6,4.0,964982224,Heat (1995),action crime thriller,1995,Heat,3.946078,102,Heat action crime thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),mystery thriller,1995,Seven (a.k.a. Se7en),3.975369,203,Seven (a.k.a. Se7en) mystery thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",crime mystery thriller,1995,"Usual Suspects, The",4.237745,204,"Usual Suspects, The crime mystery thriller"


In [19]:
ratings = pd.read_csv("../../datasets/raw/movielens/ratings.csv")

ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [20]:
user_movie_matrix = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

user_movie_matrix.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
user_movie_matrix = user_movie_matrix.fillna(0)

In [22]:
user_similarity = cosine_similarity(user_movie_matrix)

print(user_similarity.shape)

(610, 610)


In [23]:
tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(movie_data["content"])

In [24]:
content_similarity = cosine_similarity(
    tfidf_matrix,
    tfidf_matrix
)

In [25]:
movie_indices = pd.Series(
    movie_data.index,
    index=movie_data["clean_title"]
)
movie_indices = movie_indices[~movie_indices.index.duplicated(keep="first")]

In [26]:
def get_content_scores(title):

    idx = movie_indices[title]

    scores = list(
        enumerate(content_similarity[idx])
    )

    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )

    return scores

In [27]:
def collaborative_score(movie_id):

    avg_rating = ratings[
        ratings["movieId"] == movie_id
    ]["rating"].mean()

    if np.isnan(avg_rating):
        return 0

    return avg_rating / 5

In [28]:
def hybrid_recommendation(title, top_n=10):

    if title not in movie_indices:
        return "Movie not found"

    scores = get_content_scores(title)

    recommendations = []

    for index, similarity in scores[1:100]:

        movie = movie_data.iloc[index]

        collab = collaborative_score(
            movie["movieId"]
        )

        hybrid = (
            similarity * 0.7 +
            collab * 0.3
        )

        recommendations.append([

            movie["clean_title"],

            movie["genres"],

            round(similarity,3),

            round(collab,3),

            round(hybrid,3)

        ])

    recommendations = sorted(

        recommendations,

        key=lambda x:x[4],

        reverse=True

    )

    return pd.DataFrame(

        recommendations[:top_n],

        columns=[

            "Movie",

            "Genres",

            "Content Score",

            "Collaborative Score",

            "Hybrid Score"

        ]

    )

In [29]:
hybrid_recommendation("Toy Story")

,Movie,Genres,Content Score,Collaborative Score,Hybrid Score
0,Toy Story 2,adventure animation children comedy fantasy,1.000,0.772,0.932
1,Toy Story 3,adventure animation children comedy fantasy imax,0.938,0.822,0.903
2,Up,adventure animation children drama,0.459,0.801,0.562
3,"NeverEnding Story, The",adventure children fantasy,0.478,0.716,0.549
4,L.A. Story,comedy romance,0.470,0.696,0.538
5,"Christmas Story, A",children comedy,0.420,0.795,0.532
6,Inside Out,adventure animation children comedy drama fantasy,0.387,0.763,0.500
7,"Monsters, Inc.",adventure animation children comedy fantasy,0.382,0.774,0.500
8,Shrek,adventure animation children comedy fantasy ro...,0.375,0.774,0.495
9,"Story of Us, The",comedy drama,0.492,0.480,0.489


In [30]:
hybrid_recommendation("Batman")

,Movie,Genres,Content Score,Collaborative Score,Hybrid Score
0,Batman,action adventure comedy,0.801,0.600,0.741
1,Batman Returns,action crime,0.695,0.605,0.668
2,Batman Begins,action crime imax,0.598,0.772,0.651
3,Batman Forever,action adventure comedy crime,0.650,0.583,0.630
4,F/X,action crime thriller,0.555,0.644,0.582
5,Batman & Robin,action adventure fantasy thriller,0.626,0.443,0.571
6,The Lego Batman Movie,action animation comedy,0.482,0.743,0.560
7,"Batman: The Dark Knight Returns, Part 2",action animation,0.436,0.775,0.538
8,"Batman: The Dark Knight Returns, Part 1",action animation sci-fi,0.418,0.786,0.529
9,Batman: Mask of the Phantasm,animation children,0.404,0.823,0.529


In [31]:
popular_movies = ratings.groupby(
    "movieId"
)["rating"].mean().sort_values(
    ascending=False
)

popular_movies.head(20)


movieId
187717    5.0
6983      5.0
5328      5.0
95843     5.0
3941      5.0
3940      5.0
3939      5.0
7815      5.0
162414    5.0
162344    5.0
158882    5.0
53578     5.0
96430     5.0
6201      5.0
6192      5.0
158027    5.0
157775    5.0
126088    5.0
124851    5.0
124404    5.0
Name: rating, dtype: float64

In [32]:
ratings.groupby(
    "movieId"
).size().sort_values(
    ascending=False
).head(20)

movieId
356     329
318     317
296     307
593     279
2571    278
260     251
480     238
110     237
589     224
527     220
2959    218
1       215
1196    211
2858    204
50      204
47      203
780     202
150     201
1198    200
4993    198
dtype: int64

In [33]:
print("Movies :", len(movie_data))
print("Users :", ratings["userId"].nunique())
print("Ratings :", len(ratings))

Movies : 3650
Users : 610
Ratings : 100836


In [34]:
movie_data.to_csv(

    "../../datasets/processed/hybrid_dataset.csv",

    index=False

)

print("Hybrid Dataset Saved")

Hybrid Dataset Saved


In [35]:
print("="*60)

print("HYBRID RECOMMENDATION COMPLETED")

print("="*60)

HYBRID RECOMMENDATION COMPLETED
